# CRUD Operations in ChromaDB (Adding and Querying Documents)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete — the four basic operations every database supports.

### What you will learn
1. **Create** — set up a client and collection
2. **Create (Add)** — insert documents with ids and metadata
3. **Read (Query)** — semantic search + get-by-id + filters
4. **Update** — modify existing documents/metadata
5. **Delete** — remove documents by id or by filter
6. Checking collection state (`count`, `peek`, `list_collections`)

### 1. Install ChromaDB
Run this once.

In [1]:
!pip install -q chromadb

### 2. CREATE — Client and Collection

In [2]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Create a fresh collection to practice CRUD operations on
collection = client.create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: crud_demo
Current document count: 0


### 3. CREATE — Adding Documents (`add`)

Each document needs a unique `id`. `metadata` and `documents` are optional extras that describe each entry.

In [3]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

Documents added! Total count: 5


**Discussion Point:** If you try to `add()` an `id` that already exists, ChromaDB will raise an error. Use `upsert()` (shown later) if you want "add or update" behavior instead.

### 4. READ — Peek at the Collection

`peek()` shows a quick sample of what's inside, without needing a query.

In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

doc1 -> Python is a popular programming language for AI.
doc2 -> The weather today is sunny and warm.
doc3 -> Machine learning models learn patterns from data.


### 5. READ — Get Documents by ID
Use `get()` when you know exactly which id(s) you want — this is an exact lookup, not a similarity search.

In [5]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

doc1 | {'category': 'tech'} | Python is a popular programming language for AI.
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


### 6. READ — Semantic Search (`query`)

This is the core "vector database" feature: search by **meaning**, not exact words.

In [6]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc1 | distance=0.9516 | Python is a popular programming language for AI.


### 7. READ — Filtering with `where` (Metadata Filter)

Combine semantic search with exact filters — e.g., "only search within the 'tech' category".

In [7]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc4 | distance=1.3895 | I enjoy playing football on weekends.


### 8. UPDATE — Modifying Existing Documents

`update()` changes the text and/or metadata of documents that **already exist** (matched by `id`).

In [8]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

['I enjoy playing football and basketball on weekends.']
[{'category': 'sports', 'updated': True}]


### 9. UPSERT — Add or Update in One Step

`upsert()` is useful when you're not sure if the `id` already exists: if it exists, it updates it; if not, it creates it.

In [9]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",  # existing id -> will update
        "The Eiffel Tower is located in Paris."                           # new id -> will create
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

Total count after upsert: 6
['Deep learning uses neural networks with many layers and GPUs.', 'The Eiffel Tower is located in Paris.']


### 10. DELETE — Removing Documents by ID

In [10]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['doc1', 'doc3', 'doc4', 'doc5', 'doc6']


### 11. DELETE — Removing Documents by Filter

You can also delete every document that matches a metadata filter, without listing ids manually.

In [11]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 2
Remaining ids: ['doc4', 'doc6']


### 12. Verifying Everything (Final State Check)

In [12]:
final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 2
doc4 | {'updated': True, 'category': 'sports'} | I enjoy playing football and basketball on weekends.
doc6 | None | The Eiffel Tower is located in Paris.


### 13. Key Takeaways (Recap for Students)

| Operation | Method | Notes |
|---|---|---|
| Create | `client.create_collection()` | Makes a new collection (errors if it already exists) |
| Create/Add | `collection.add()` | Insert new documents; errors on duplicate `id` |
| Read (exact) | `collection.get()` | Fetch by `id`, no similarity involved |
| Read (semantic) | `collection.query()` | Search by meaning, returns closest matches + distances |
| Read (preview) | `collection.peek()` | Quick look at a few stored documents |
| Update | `collection.update()` | Modify existing document/metadata by `id` |
| Add or Update | `collection.upsert()` | Creates if missing, updates if it exists |
| Delete | `collection.delete()` | Remove by `id` list or by `where` metadata filter |

### 14. Practice Exercises
1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new one in a single call.
2. Delete all documents where `category` equals a value of your choice, then confirm with `collection.count()`.
3. Write a small function `safe_add(collection, id, text)` that uses `upsert()` internally so it never raises a duplicate-id error.

In [13]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [14]:
import chromadb

client = chromadb.Client()

# Remove old collection if it exists, so the program can be run again
try:
    client.delete_collection("practice1")
except:
    pass

collection = client.create_collection("practice1")

# Add 5 documents
collection.add(
    ids=["1", "2", "3", "4", "5"],
    documents=[
        "Python programming",
        "Java programming",
        "MySQL database",
        "HTML and CSS",
        "Data Structures"
    ]
)

# Update 2 existing documents and create 1 new document
collection.upsert(
    ids=["2", "4", "6"],
    documents=[
        "Advanced Java programming",
        "Advanced HTML and CSS",
        "Machine Learning"
    ]
)

print("All documents:")
print(collection.get())

print("\nTotal documents:", collection.count())

All documents:
{'ids': ['1', '2', '3', '4', '5', '6'], 'embeddings': None, 'documents': ['Python programming', 'Advanced Java programming', 'MySQL database', 'Advanced HTML and CSS', 'Data Structures', 'Machine Learning'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None, None, None]}

Total documents: 6


In [15]:
import chromadb

client = chromadb.Client()

try:
    client.delete_collection("practice2")
except:
    pass

collection = client.create_collection("practice2")

# Add documents with categories
collection.add(
    ids=["1", "2", "3", "4", "5"],
    documents=[
        "Laptop",
        "Mobile Phone",
        "Python Book",
        "Headphones",
        "Java Book"
    ],
    metadatas=[
        {"category": "electronics"},
        {"category": "electronics"},
        {"category": "books"},
        {"category": "electronics"},
        {"category": "books"}
    ]
)

print("Before deletion:", collection.count())

# Delete all electronics
collection.delete(
    where={"category": "electronics"}
)

print("After deletion:", collection.count())

print("\nRemaining documents:")
print(collection.get())

Before deletion: 5
After deletion: 2

Remaining documents:
{'ids': ['3', '5'], 'embeddings': None, 'documents': ['Python Book', 'Java Book'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'books'}, {'category': 'books'}]}


In [1]:
import chromadb

client = chromadb.Client()

try:
    client.delete_collection("practice3")
except:
    pass

collection = client.create_collection("practice3")


def safe_add(collection, id, text):
    collection.upsert(
        ids=[id],
        documents=[text]
    )


safe_add(collection, "1", "Python Programming")
safe_add(collection, "2", "Java Programming")

# Same ID again → updates instead of error
safe_add(collection, "1", "Advanced Python Programming")

print(collection.get())

{'ids': ['1', '2'], 'embeddings': None, 'documents': ['Advanced Python Programming', 'Java Programming'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None]}
